# Personal SLM Chatbot

This notebook builds a small chatbot that answers questions about you from a text file called `my_info.txt`.

It uses a Retrieval Augmented Generation (RAG) flow:
1. Read your personal notes from `my_info.txt`
2. Split them into chunks by double newlines
3. Embed the chunks with `all-MiniLM-L6-v2`
4. Store them in ChromaDB
5. Retrieve the top 1-2 relevant chunks for a question
6. Use `google/flan-t5-base` to answer using only retrieved chunks
7. Launch a Gradio chat UI with example question buttons

Run the notebook from top to bottom after you create `my_info.txt`.

## What is RAG?

Retrieval Augmented Generation, or RAG, is a pattern where the model first retrieves relevant context from a knowledge base and then generates an answer from that context.

For this project, your file is the knowledge base, ChromaDB stores the vectors, and the language model only sees the retrieved chunks before it answers.

## Why use a small LM instead of GPT-4?

A small model like `google/flan-t5-base` is a great choice for a portfolio project because it is lighter, cheaper to experiment with, easier to deploy on a free Hugging Face Space, and it demonstrates the full RAG pipeline clearly.

GPT-4 is more capable, but the goal here is to show a reproducible personal chatbot that can run locally and be shared publicly without heavy infrastructure.

In [125]:
import sys
!{sys.executable} -m pip install --upgrade --index-url https://download.pytorch.org/whl/cpu torch torchvision torchaudio

Looking in indexes: https://download.pytorch.org/whl/cpu



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [126]:
# Install dependencies
import sys
import subprocess
import importlib.util

packages = [
    "sentence-transformers",
    "chromadb",
    "transformers",
    "gradio",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg])

# Torch often needs the PyTorch CPU wheel index in notebook environments.
# We try a known-good CPU wheel first, then fall back to the default index.
torch_install_commands = [
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--index-url", "https://download.pytorch.org/whl/cpu", "torch", "torchvision", "torchaudio"],
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torch", "torchvision", "torchaudio"],
]

torch_installed = False
last_error = None
for cmd in torch_install_commands:
    try:
        subprocess.check_call(cmd)
        torch_installed = True
        break
    except Exception as e:
        last_error = e

if torch_installed:
    print("Dependencies installed.")
else:
    print("Torch installation warning:", last_error)
    print("If needed, run this manually in a cell:")
    print("python -m pip install --upgrade --index-url https://download.pytorch.org/whl/cpu torch torchvision torchaudio")

Dependencies installed.


In [127]:
# Verify installation
required_modules = ["sentence_transformers", "chromadb", "transformers", "gradio", "torch"]
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
if missing:
    message = f"Missing modules after install: {missing}"
    if "torch" in missing:
        message += "\nTorch is required for the FLAN-T5 generation step. Install it with:\n"
        message += "python -m pip install --upgrade --index-url https://download.pytorch.org/whl/cpu torch torchvision torchaudio"
    raise ImportError(message)

import json
from pathlib import Path
import torch
import chromadb
import gradio as gr
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("All required libraries imported successfully.")
print("Torch version:", torch.__version__)

All required libraries imported successfully.
Torch version: 2.12.0+cpu


In [128]:
# Git commands for your repo workflow
print(r'''Git commands:
1. Initialize repo:
   git init

2. Add files:
   git add .

3. Commit:
   git commit -m "Build personal SLM chatbot"

4. Push to GitHub:
   git remote add origin <your-github-repo-url>
   git branch -M main
   git push -u origin main
''')

Git commands:
1. Initialize repo:
   git init

2. Add files:
   git add .

3. Commit:
   git commit -m "Build personal SLM chatbot"

4. Push to GitHub:
   git remote add origin <your-github-repo-url>
   git branch -M main
   git push -u origin main



In [129]:
# Configuration
DATA_FILE = Path("my_info.txt")
CHROMA_DIR = Path("chroma_db")
ARTIFACT_DIR = Path("chatbot_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
GENERATION_MODEL_NAME = "google/flan-t5-base"
TOP_K = 3
MAX_NEW_TOKENS = 128
NO_INFO_MSG = "Move to contact form for all details."
GREETING_RESPONSE = (
    "Hello! I can help with my background, skills, services, projects, and availability. "
    "Feel free to ask for a short portfolio summary, and if you need full project or collaboration details, move to contact form for all details."
)
CONTACT_FALLBACK = "Move to contact form for all details."

print("Configuration loaded.")

Configuration loaded.


## Load the source text

The notebook expects `my_info.txt` in the same folder. It handles file-not-found and empty-file cases with clear errors.

In [130]:
def load_text_file(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f"Could not find {path}. Create it in the same folder as this notebook.")
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError(f"{path} is empty. Please add your personal information.")
    return text

raw_text = load_text_file(DATA_FILE)
print("Loaded text length:", len(raw_text))

Loaded text length: 4566


## Split into chunks

We split on double newlines exactly as requested and remove empty chunks.

In [131]:
def split_into_chunks(text: str):
    chunks = [chunk.strip() for chunk in text.split("\n\n")]
    chunks = [chunk for chunk in chunks if chunk]
    if not chunks:
        raise ValueError("No valid chunks were found after splitting the text.")
    return chunks

chunks = split_into_chunks(raw_text)
print(f"Created {len(chunks)} chunks.")
for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\nChunk {i}:\n{chunk[:250]}")

Created 23 chunks.

Chunk 1:
BIO

Chunk 2:
My name is Zyad Wael Mohamed Khedr. I am a Data Science and Mobile Software Engineering student at Alexandria University's Faculty of Computers and Data Science. My expected graduation date is October 2027.

Chunk 3:
Since 2023, I have focused on building high-performance cross-platform mobile applications using Flutter. I care about Clean Architecture, SOLID principles, intuitive UI and UX, Material 3, and the 8-point grid rule. I enjoy turning complex requireme


## Build embeddings and store them in ChromaDB

The embedding model converts each chunk into a vector, and ChromaDB stores the vectors for retrieval later.

In [132]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Loaded embedding model: {EMBEDDING_MODEL_NAME}")

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(name="personal_info")

try:
    existing = collection.count()
    if existing > 0:
        collection.delete(where={})
        print(f"Cleared {existing} existing records from ChromaDB.")
except Exception as e:
    print("ChromaDB cleanup note:", e)

ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [{"source": "my_info.txt", "chunk_index": i} for i in range(len(chunks))]
embeddings = embedding_model.encode(chunks, normalize_embeddings=True).tolist()

collection.add(
    ids=ids,
    documents=chunks,
    embeddings=embeddings,
    metadatas=metadatas,
)

print(f"Stored {collection.count()} chunks in ChromaDB at {CHROMA_DIR}.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8584.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedding model: all-MiniLM-L6-v2
ChromaDB cleanup note: Expected where to have exactly one operator, got {} in delete.
Stored 23 chunks in ChromaDB at chroma_db.


## Test retrieval before adding the language model

This step is important because it confirms the vector search is working before we add generation.

In [133]:
def retrieve_chunks(query: str, top_k: int = TOP_K):
    if not query or not query.strip():
        raise ValueError("Question cannot be empty.")
    query_embedding = embedding_model.encode([query], normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "distances", "metadatas"],
    )
    docs = results.get("documents", [[]])[0]
    distances = results.get("distances", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    retrieved = []
    for doc, dist, meta in zip(docs, distances, metadatas):
        retrieved.append({"text": doc, "distance": dist, "metadata": meta})
    return retrieved

test_question = "What is my educational background?"
retrieved = retrieve_chunks(test_question, top_k=TOP_K)

if not retrieved:
    print("No results found.")
else:
    print(f"Retrieved {len(retrieved)} chunks for: {test_question}\n")
    for i, item in enumerate(retrieved, start=1):
        print(f"Result {i} | distance={item['distance']:.4f} | metadata={item['metadata']}")
        print(item["text"])
        print("-" * 80)

Retrieved 3 chunks for: What is my educational background?

Result 1 | distance=1.2038 | metadata={'source': 'my_info.txt', 'chunk_index': 0}
BIO:
My name is Zyad Wael Mohamed Khedr. I am a Data Science and Mobile Software 
Engineering student at Alexandria University's Faculty of Computers and Data 
Science, with an expected graduation date of October 2027.
--------------------------------------------------------------------------------
Result 2 | distance=1.4321 | metadata={'source': 'my_info.txt', 'chunk_index': 12}
EXPERIENCE
--------------------------------------------------------------------------------
Result 3 | distance=1.4574 | metadata={'chunk_index': 14, 'source': 'my_info.txt'}
I also worked as a Flutter Developer Intern at Cellula Technologies from January 2025 to April 2025. There I built intelligent meal recommendation engines, AI weather pattern analyzers, and worked with asynchronous data streams, state management pipelines, and Firebase integrations.
----------------

## Load the small language model

We use `google/flan-t5-base` to write a natural answer using only the retrieved chunks.

In [134]:
tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Loaded generation model: {GENERATION_MODEL_NAME} on {device}")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 7665.40it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded generation model: google/flan-t5-base on cpu


## Answer generation

If retrieval is weak or returns nothing useful, the bot should not invent details. It should safely say that the answer is not in the file.

In [135]:
def build_context(retrieved_items):
    if not retrieved_items:
        return ""
    retrieved_items = sorted(retrieved_items, key=lambda item: item["distance"])
    parts = []
    for idx, item in enumerate(retrieved_items, start=1):
        parts.append(f"[Chunk {idx} | distance={item['distance']:.4f}]\n{item['text']}")
    return "\n\n".join(parts)


def detect_intent(question: str) -> str:
    q = question.lower().strip()
    if any(word in q for word in ["hi", "hello", "hey", "good morning", "good afternoon", "good evening"]):
        return "greeting"
    if any(word in q for word in ["service", "services", "what can you do", "what do you do", "can you do", "capabilities"]):
        return "services"
    if any(word in q for word in ["availability", "available", "freelance", "work with", "collaboration", "hire"]):
        return "availability"
    if any(word in q for word in ["skill", "skills", "technologies", "tech stack", "tools"]):
        return "skills"
    if any(word in q for word in ["study", "studied", "education", "university", "college", "school"]):
        return "education"
    if any(word in q for word in ["project", "projects", "built", "worked on", "experience", "current experience", "work experience", "professional experience"]):
        return "projects"
    if any(word in q for word in ["goal", "goals", "career", "future"]):
        return "goals"
    return "general"


def answer_question(question: str):
    try:
        retrieved_items = retrieve_chunks(question, top_k=TOP_K)
    except Exception as e:
        return f"Error during retrieval: {e}"

    intent = detect_intent(question)

    if intent == "greeting":
        return GREETING_RESPONSE
    if intent == "services":
        return (
            "I offer Flutter mobile and desktop app development, clean app architecture, AI chatbot and recommendation features, backend integration with Supabase, Firebase, REST APIs, and WebSockets, plus UI and UX design, optimization, hardware integrations, monetization, and app deployment. "
            "For project-specific details, move to contact form for all details."
        )
    if intent == "availability":
        return (
            "I am available for selected freelance or collaboration projects depending on scope, timeline, and fit. "
            "For project inquiries, timelines, pricing, and collaboration details, move to contact form for all details."
        )
    if intent == "skills":
        return (
            "My main skills include Flutter, Dart, Python, Java, Swift, SwiftUI, Riverpod, Provider, Cubit, Supabase Realtime, Firebase, Node.js, REST APIs, WebSockets, SQLite, Gemini AI, TensorFlow, Git, GitHub, CMake, Google Maps, and Google AdMob."
        )
    if intent == "education":
        return (
            "I am a Data Science and Mobile Software Engineering student at Alexandria University's Faculty of Computers and Data Science, with an expected graduation date of October 2027."
        )
    if intent == "goals":
        return (
            "My short-term goal is to deploy this chatbot on my portfolio website. My long-term goal is to work in Mobile Software Engineering, Cross-Platform Architecture, or AI-Driven Application Development after graduation."
        )
    if intent == "projects":
        return (
            "I have worked on Payss, a social loyalty and rewards app; Droplet, a smart rainwater harvesting app with AI and IoT features; MazoBoothMirror, a B2B photo booth desktop app; Liquid Navbar, an open-source Flutter UI package; Iron Incognito, a privacy-first social gym app; and Klaket, a multiplayer acting and guessing game. For specific project details, move to contact form for all details."
        )
    if intent == "general" and any(word in question.lower().strip() for word in ["experience", "current experience", "work experience", "professional experience"]):
        return (
            "My current experience is centered on Flutter development, cross-platform apps, clean architecture, AI integrations, and production releases. I have worked at ZeroOneZ as a Flutter Developer and at Cellula Technologies as a Flutter Developer Intern."
        )

    if not retrieved_items:
        return CONTACT_FALLBACK

    best_distance = min(item["distance"] for item in retrieved_items)
    print(f"Retrieved {len(retrieved_items)} chunks | best distance={best_distance:.4f}")
    for item in retrieved_items:
        print(f"- distance={item['distance']:.4f} | {item['text'][:120].replace(chr(10), ' ')}")

    context = build_context(retrieved_items)
    prompt = f'''You are a careful personal assistant chatbot.
Answer the user's question using ONLY the context below.
Rules:
- Do not use outside knowledge.
- Do not guess or invent facts.
- If the answer is not clearly in the context, reply exactly: "{CONTACT_FALLBACK}"
- Keep the answer concise, natural, and specific.
- If the context contains multiple relevant facts, combine them into 1-3 short sentences.
- If the user asks for a list, use bullets only when the context supports it.
- If the question is about services, availability, pricing, timelines, or collaboration details and the context does not contain a direct answer, respond with: "{CONTACT_FALLBACK}"

Context:
{context}

Question:
{question}

Answer:
'''.strip()

    try:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                repetition_penalty=1.1,
            )
        result = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
        return result or CONTACT_FALLBACK
    except Exception as e:
        return f"Generation error: {e}"

print("Answer helper ready.")

Answer helper ready.


## Quick chatbot tests

Try a few sample questions before launching the interface.

In [136]:
sample_questions = [
    "What is my name?",
    "Where did I study?",
    "What are my hobbies?",
]

for q in sample_questions:
    print(f"Q: {q}")
    print(f"A: {answer_question(q)}")
    print("-" * 80)

Q: What is my name?
Retrieved 3 chunks | best distance=1.2474
- distance=1.2474 | BIO: My name is Zyad Wael Mohamed Khedr. I am a Data Science and Mobile Software  Engineering student at Alexandria Univ
- distance=1.5184 | SERVICES
- distance=1.5700 | [PROJECT 5] Iron Incognito – Social Gym Application * Description: A multi-tenant, privacy-first fitness social applicat
A: Zyad Wael Mohamed Khedr.
--------------------------------------------------------------------------------
Q: Where did I study?
A: I am a Data Science and Mobile Software Engineering student at Alexandria University's Faculty of Computers and Data Science, with an expected graduation date of October 2027.
--------------------------------------------------------------------------------
Q: What are my hobbies?
Retrieved 3 chunks | best distance=1.4805
- distance=1.4805 | GOALS
- distance=1.4807 | SERVICES
- distance=1.5092 | SKILLS: * Frameworks & Core Architecture: Flutter SDK (Mobile & Desktop), Flame Engine,        

## Save the chatbot so it can be reused

This saves the important settings and keeps the ChromaDB store on disk so you can reload it later.

In [137]:
def save_artifacts():
    metadata = {
        "embedding_model": EMBEDDING_MODEL_NAME,
        "generation_model": GENERATION_MODEL_NAME,
        "top_k": TOP_K,
        "max_new_tokens": MAX_NEW_TOKENS,
        "source_file": str(DATA_FILE),
        "chroma_dir": str(CHROMA_DIR),
    }
    (ARTIFACT_DIR / "chatbot_config.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"Saved chatbot config to {ARTIFACT_DIR / 'chatbot_config.json'}")
    print(f"Vector DB is persisted in {CHROMA_DIR}")

save_artifacts()

Saved chatbot config to chatbot_artifacts\chatbot_config.json
Vector DB is persisted in chroma_db


## Gradio chat interface

The interface includes example question buttons and a simple chat window.

In [138]:
EXAMPLE_QUESTIONS = [
    "Hi, what can you help me with?",
    "Give me a portfolio summary",
    "What services do I offer?",
    "Am I available for projects?",
]

def is_greeting(message: str) -> bool:
    text = message.lower().strip()
    greeting_words = ["hi", "hello", "hey", "good morning", "good afternoon", "good evening"]
    return any(text == word or text.startswith(word + " ") for word in greeting_words)


def chat_fn(message, history):
    history = history or []
    if is_greeting(message):
        response = GREETING_RESPONSE
    else:
        response = answer_question(message)
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response},
    ]
    return history, history

with gr.Blocks(title="Personal SLM Chatbot") as demo:
    gr.Markdown("# Personal SLM Chatbot\nAsk questions about the information in your `my_info.txt` file.")
    gr.Markdown("Hello! Ask me for a portfolio summary, background, skills, services, projects, or availability for collaboration.")
    chatbot = gr.Chatbot(label="Chat", height=450)
    state = gr.State([])
    msg = gr.Textbox(label="Your question", placeholder="Ask something about you...")
    clear = gr.Button("Clear chat")
    gr.Markdown("### Example questions")
    with gr.Row():
        btn1 = gr.Button(EXAMPLE_QUESTIONS[0])
        btn2 = gr.Button(EXAMPLE_QUESTIONS[1])
    with gr.Row():
        btn3 = gr.Button(EXAMPLE_QUESTIONS[2])
        btn4 = gr.Button(EXAMPLE_QUESTIONS[3])

    btn1.click(fn=lambda: EXAMPLE_QUESTIONS[0], inputs=None, outputs=msg)
    btn2.click(fn=lambda: EXAMPLE_QUESTIONS[1], inputs=None, outputs=msg)
    btn3.click(fn=lambda: EXAMPLE_QUESTIONS[2], inputs=None, outputs=msg)
    btn4.click(fn=lambda: EXAMPLE_QUESTIONS[3], inputs=None, outputs=msg)

    msg.submit(chat_fn, inputs=[msg, state], outputs=[chatbot, state])
    clear.click(lambda: ([], []), None, [chatbot, state])

demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


Retrieved 3 chunks | best distance=1.5206
- distance=1.5206 | GOALS: Short-Term Goal: Deploying this tailored Small Language Model (SLM) directly into    my personal interactive port
- distance=1.5707 | I can also help with app optimization, state management, hardware integrations, Google Maps features, monetization with 
- distance=1.5783 | BIO: My name is Zyad Wael Mohamed Khedr. I am a Data Science and Mobile Software  Engineering student at Alexandria Univ
Retrieved 3 chunks | best distance=1.3624
- distance=1.3624 | BIO: My name is Zyad Wael Mohamed Khedr. I am a Data Science and Mobile Software  Engineering student at Alexandria Univ
- distance=1.4773 | AVAILABILITY
- distance=1.5111 | SERVICES
Retrieved 3 chunks | best distance=1.5373
- distance=1.5373 | AVAILABILITY
- distance=1.5775 | SERVICES
- distance=1.5858 | [PROJECT 6] Klaket – Mobile Acting & Guessing Game * Description: A dynamic, multiplayer party and local acting game bui


## Deploying to Hugging Face Spaces

To deploy for free:
1. Create a Hugging Face account
2. Create a new Space
3. Choose `Gradio`
4. Convert this notebook logic into an `app.py` Gradio app or export the working code into a script
5. Add `requirements.txt`
6. Push the repo to the Space

Gradio is supported natively, so this project is a good fit for the free tier.

## `my_info.txt` format

Use double newlines to separate topics. Each paragraph becomes one chunk.

For better answers, make the file concrete and detailed. Include:
- your full name
- education
- skills and tools
- projects with short descriptions
- hobbies and interests
- goals and career plans
- languages you speak
- internships, clubs, or awards

Example:
```text
My name is Alex Johnson. I am a data science student based in Cairo.

I study computer science at Example University and I am interested in machine learning, NLP, and data analysis.

My skills include Python, pandas, NumPy, SQL, scikit-learn, TensorFlow, Git, and Jupyter notebooks.

I have worked on a movie recommendation system, a customer churn prediction project, and a personal chatbot project.

My hobbies include reading, football, fitness, and building small AI tools.

My goal is to become a machine learning engineer and build useful AI products.
```

Keep each paragraph focused on one topic and write facts you want the bot to answer about. The more specific the text, the better the answers.

## Running locally

1. Put `my_info.txt` in the same folder as this notebook
2. Run all cells from top to bottom
3. Ask questions in the Gradio interface
4. Keep the `chroma_db` folder if you want to reuse the stored vectors later

## Free Hugging Face Spaces deployment

1. Create a Hugging Face account
2. Create a new Space
3. Choose Gradio
4. Add `app.py`, `requirements.txt`, and your project files
5. Make sure the app can access `my_info.txt`
6. Push to the Space

If you want, you can later convert this notebook into a standalone `app.py` file for a cleaner deployment.

## Final reminders

Git commands:
```bash
git init
git add .
git commit -m "Build personal SLM chatbot"
git remote add origin <your-github-repo-url>
git branch -M main
git push -u origin main
```

This notebook is now ready to run after you create `my_info.txt`.